# Scanner de Temas Emergentes v7

| Celda | Que hace | Cuando |
|-------|----------|--------|
| 1 | Instala librerias | Solo la primera vez |
| 2 | Carga el codigo | SIEMPRE antes de 3 o 4 |
| 3 | Watchlist personal | ~30 seg |
| 4 | S&P 500 completo | ~4 min |

Novedades v7: historial de scores (7 dias) + evolucion + rotacion de temas

In [ ]:
!pip install yfinance pandas requests beautifulsoup4 -q
print('OK librerias instaladas')

In [ ]:
# CELDA 2 — Ejecutar SIEMPRE antes de la 3 o la 4
import math, warnings, json, base64
from datetime import datetime
from pathlib import Path
import numpy as np
import pandas as pd
import requests
import yfinance as yf
warnings.filterwarnings('ignore')

GITHUB_USER  = 'Carolo-III'
GITHUB_REPO  = 'scanner-temas'
GITHUB_TOKEN = ''  # <-- PON AQUI TU TOKEN

PERSONAL_WATCHLIST = {
    'Semiconductores': ['ARM','AMD','MU','MTSI','POET','SMCI'],
    'Infraestructura AI': ['ANET','VRT','APLD','CORZ','IREN','CIFR','CRWV'],
    'Espacio y Defensa': ['RKLB','LUNR','ASTS','KTOS','BWXT'],
    'Cuantica': ['IONQ'],
    'Robotica AI': ['BBAI','TSLA'],
    'Crypto Fintech': ['RDDT'],
    'Minerales': ['MP','UAMY'],
    'Biotech': ['VKTX','ACRV','IBRX'],
    'Hardware': ['SNDK'],
    'Momentum': ['KOPN','ONDS','BE','LITE','GILT'],
}

SECTOR_LABELS = {
    'Technology':'Tecnologia','Health Care':'Salud','Financials':'Finanzas',
    'Consumer Discretionary':'Consumo Discrecional','Industrials':'Industriales',
    'Communication Services':'Comunicacion','Consumer Staples':'Consumo Basico',
    'Energy':'Energia','Utilities':'Utilities','Real Estate':'Real Estate','Materials':'Materiales',
}

def download_prices(tickers, period='6mo'):
    if not tickers:
        return pd.DataFrame(), pd.DataFrame()
    LOTE = 50
    all_c, all_v = [], []
    lotes = [tickers[i:i+LOTE] for i in range(0, len(tickers), LOTE)]
    for i, lote in enumerate(lotes):
        if len(lotes) > 1:
            print('    Lote ' + str(i+1) + '/' + str(len(lotes)) + '...', end=' ')
        try:
            raw = yf.download(lote, period=period, interval='1d',
                              auto_adjust=True, progress=False, threads=True)
            if raw.empty:
                if len(lotes) > 1: print('vacio')
                continue
            if isinstance(raw.columns, pd.MultiIndex):
                c, v = raw['Close'], raw['Volume']
            else:
                c = raw[['Close']]; c.columns = lote[:1]
                v = raw[['Volume']]; v.columns = lote[:1]
            all_c.append(c); all_v.append(v)
            if len(lotes) > 1: print('OK')
        except Exception as e:
            if len(lotes) > 1: print('error: ' + str(e))
    if not all_c:
        return pd.DataFrame(), pd.DataFrame()
    return pd.concat(all_c, axis=1), pd.concat(all_v, axis=1)

def rs_score(t, b, w):
    try:
        t, b = t.dropna(), b.dropna()
        if len(t) < w or len(b) < w: return None
        return round((t.iloc[-1]/t.iloc[-w]-1)*100 - (b.iloc[-1]/b.iloc[-w]-1)*100, 2)
    except: return None

def volume_zscore(v, w=20):
    try:
        v = v.dropna()
        if len(v) < w+1: return None
        m, s = v.iloc[-(w+1):-1].mean(), v.iloc[-(w+1):-1].std()
        return round((v.iloc[-1]-m)/s, 2) if s else 0.0
    except: return None

def detect_breakout(p, v, lb=50, vm=1.4):
    try:
        p, v = p.dropna(), v.dropna()
        if len(p) < lb: return {'breakout': False, 'days_ago': None}
        for d in range(1, 11):
            bh  = p.iloc[-(lb+d):-(d+5)].max()
            bvm = v.iloc[-(lb+d):-(d+5)].mean()
            if p.iloc[-d] > bh and v.iloc[-d] > vm*bvm:
                return {'breakout': True, 'days_ago': d}
        return {'breakout': False, 'days_ago': None}
    except: return {'breakout': False, 'days_ago': None}

def ma_health(p):
    try:
        p = p.dropna()
        result = {'ma20': False, 'ma50': False, 'ma200': False}
        c = float(p.iloc[-1])
        if len(p) >= 20:
            result['ma20']  = bool(c > float(p.rolling(20).mean().iloc[-1]))
        if len(p) >= 50:
            result['ma50']  = bool(c > float(p.rolling(50).mean().iloc[-1]))
        if len(p) >= 200:
            result['ma200'] = bool(c > float(p.rolling(200).mean().iloc[-1]))
        return result
    except: return {'ma20': False, 'ma50': False, 'ma200': False}

def composite_score(r4, r13, vz, bo, ma):
    s = 0
    if r4  is not None: s += 30 * min(1, max(0, (r4+30)/60))
    if r13 is not None: s += 20 * min(1, max(0, (r13+50)/100))
    if vz  is not None: s += 25 * min(1, max(0, (vz+1)/4))
    if bo: s += 15
    if ma: s += 10 * (sum([ma.get('ma20',False), ma.get('ma50',False), ma.get('ma200',False)])/3)
    return round(s, 1)

def analyze_universe(grps, bench, close_df, vol_df):
    res = []
    for gn, tickers in grps.items():
        for tk in tickers:
            if tk not in close_df.columns: continue
            p   = close_df[tk]
            v   = vol_df[tk] if tk in vol_df.columns else pd.Series(dtype=float)
            r4  = rs_score(p, bench, 20)
            r13 = rs_score(p, bench, 65)
            vz  = volume_zscore(v) if not v.empty else None
            bi  = detect_breakout(p, v) if not v.empty else {'breakout':False,'days_ago':None}
            mah = ma_health(p)
            sc  = composite_score(r4, r13, vz, bi['breakout'], mah)
            res.append({
                'ticker': tk, 'group': gn,
                'rs_4w': r4, 'rs_13w': r13, 'vol_z': vz,
                'breakout': bi['breakout'], 'days_ago': bi['days_ago'],
                'ma20': mah.get('ma20', False),
                'ma50': mah.get('ma50', False),
                'ma200': mah.get('ma200', False),
                'score': sc,
            })
    return res

def calc_groups(res, is_sp=False):
    gs = {}
    for r in res:
        gs.setdefault(r['group'], []).append(r)
    out = []
    for gn, mb in gs.items():
        scores = [m['score'] for m in mb if m['score'] is not None]
        r4s    = [m['rs_4w'] for m in mb if m['rs_4w'] is not None]
        mb_sorted = sorted(mb, key=lambda x: x['score'] or 0, reverse=True)
        out.append({
            'group':    gn,
            'score':    round(np.mean(scores), 1) if scores else 0,
            'rs_mean':  round(np.mean(r4s), 1) if r4s else 0,
            'breakouts': sum(1 for m in mb if m['breakout']),
            'n':        len(mb),
            'is_sp':    is_sp,
            'top3':     mb_sorted[:3],
            'members':  mb_sorted,
        })
    return sorted(out, key=lambda x: x['score'], reverse=True)

def get_github_file(filename, token, user, repo):
    url = 'https://api.github.com/repos/' + user + '/' + repo + '/contents/' + filename
    headers = {'Authorization': 'token ' + token}
    r = requests.get(url, headers=headers)
    if r.status_code == 200:
        content = base64.b64decode(r.json()['content']).decode('utf-8')
        return json.loads(content), r.json()['sha']
    return None, None

def upload_to_github(filename, content, token, user, repo):
    url = 'https://api.github.com/repos/' + user + '/' + repo + '/contents/' + filename
    headers = {'Authorization': 'token ' + token, 'Content-Type': 'application/json'}
    r = requests.get(url, headers=headers)
    sha = r.json().get('sha') if r.status_code == 200 else None
    if isinstance(content, str):
        content_b64 = base64.b64encode(content.encode('utf-8')).decode('utf-8')
    else:
        content_b64 = base64.b64encode(content).decode('utf-8')
    payload = {
        'message': 'Actualizar ' + filename + ' - ' + datetime.now().strftime('%d/%m/%Y %H:%M'),
        'content': content_b64,
    }
    if sha:
        payload['sha'] = sha
    r = requests.put(url, headers=headers, json=payload)
    if r.status_code in [200, 201]:
        print('  OK ' + filename + ' subido')
    else:
        print('  Error ' + filename + ': ' + str(r.status_code))

def update_history(all_groups, token, user, repo):
    today = datetime.now().strftime('%Y-%m-%d')
    history, _ = get_github_file('history.json', token, user, repo)
    if history is None:
        history = []
    # Eliminar entrada de hoy si ya existe
    history = [h for h in history if h['date'] != today]
    # Añadir entrada de hoy
    entry = {
        'date': today,
        'scores': {g['group']: g['score'] for g in all_groups},
        'ranks':  {g['group']: i+1 for i, g in enumerate(all_groups)},
    }
    history.append(entry)
    # Mantener solo los ultimos 7 dias
    history = sorted(history, key=lambda x: x['date'])[-7:]
    return history

print('OK Codigo cargado')


In [ ]:
# CELDA 3 — Watchlist personal (~30 seg)
import warnings, json
from datetime import datetime
warnings.filterwarnings('ignore')

if not GITHUB_TOKEN:
    print('ERROR: Pon tu token en GITHUB_TOKEN en la Celda 2')
else:
    print('SPY...')
    bc, _ = download_prices(['SPY'], period='6mo')
    bs = bc['SPY']
    print('  OK')

    pt = list(set(t for grp in PERSONAL_WATCHLIST.values() for t in grp))
    print('Watchlist (' + str(len(pt)) + ' valores)...')
    cp, vp = download_prices(pt + ['SPY'], period='6mo')
    print('  OK')

    print('Analizando...')
    res = analyze_universe(PERSONAL_WATCHLIST, bs, cp, vp)
    gs  = calc_groups(res, is_sp=False)
    ts  = datetime.now().strftime('%d/%m/%Y %H:%M')

    print('Actualizando historial...')
    history = update_history(gs, GITHUB_TOKEN, GITHUB_USER, GITHUB_REPO)

    data = {
        'timestamp': ts,
        'mode': 'Watchlist personal',
        'groups': gs,
        'values': sorted(res, key=lambda x: x['score'] or 0, reverse=True),
    }

    print('Subiendo a GitHub...')
    upload_to_github('data.json',    json.dumps(data,    ensure_ascii=False), GITHUB_TOKEN, GITHUB_USER, GITHUB_REPO)
    upload_to_github('history.json', json.dumps(history, ensure_ascii=False), GITHUB_TOKEN, GITHUB_USER, GITHUB_REPO)

    print('')
    print('=== RANKING ===')
    for i, g in enumerate(gs):
        bo = ' UP x' + str(g['breakouts']) if g['breakouts'] else ''
        print('#' + str(i+1) + '  ' + str(g['score']) + '  ' + g['group'] + bo)
    bos = [r for r in res if r['breakout']]
    print('Rupturas: ' + str(len(bos)))
    print('')
    print('Web actualizada: https://Carolo-III.github.io/scanner-temas')


In [ ]:
# CELDA 4 — S&P 500 completo (~4 min)
import warnings, json, pandas as pd
from datetime import datetime
warnings.filterwarnings('ignore')

if not GITHUB_TOKEN:
    print('ERROR: Pon tu token en GITHUB_TOKEN en la Celda 2')
else:
    print('SPY...')
    bc, _ = download_prices(['SPY'], period='6mo')
    bs = bc['SPY']
    print('  OK')

    pt = list(set(t for grp in PERSONAL_WATCHLIST.values() for t in grp))
    print('Watchlist (' + str(len(pt)) + ' valores)...')
    cp, vp = download_prices(pt + ['SPY'], period='6mo')
    pr  = analyze_universe(PERSONAL_WATCHLIST, bs, cp, vp)
    pgs = calc_groups(pr, is_sp=False)
    print('  OK ' + str(len(pr)) + ' valores')

    print('Lista S&P 500...')
    url = 'https://raw.githubusercontent.com/datasets/s-and-p-500-companies/main/data/constituents.csv'
    df_sp = pd.read_csv(url)
    sbs = {}
    for _, row in df_sp.iterrows():
        tk = row['Symbol'].replace('.', '-')
        if tk not in pt:
            lb = SECTOR_LABELS.get(row['GICS Sector'], row['GICS Sector'])
            sbs.setdefault(lb, []).append(tk)

    sa = list(set(t for grp in sbs.values() for t in grp))
    print('Descargando ' + str(len(sa)) + ' valores en lotes...')
    cs, vs = download_prices(sa, period='6mo')
    sr  = analyze_universe(sbs, bs, cs, vs)
    sgs = calc_groups(sr, is_sp=True)
    print('  OK ' + str(len(sr)) + ' valores')

    ar = pr + sr
    ts = datetime.now().strftime('%d/%m/%Y %H:%M')
    all_groups = sorted(pgs + sgs, key=lambda x: x['score'], reverse=True)

    print('Actualizando historial...')
    history = update_history(all_groups, GITHUB_TOKEN, GITHUB_USER, GITHUB_REPO)

    data = {
        'timestamp': ts,
        'mode': 'S&P500 + Watchlist',
        'groups': all_groups,
        'values': sorted(ar, key=lambda x: x['score'] or 0, reverse=True),
    }

    print('Subiendo a GitHub...')
    upload_to_github('data.json',    json.dumps(data,    ensure_ascii=False), GITHUB_TOKEN, GITHUB_USER, GITHUB_REPO)
    upload_to_github('history.json', json.dumps(history, ensure_ascii=False), GITHUB_TOKEN, GITHUB_USER, GITHUB_REPO)

    todos = all_groups
    print('')
    print('=== TOP 5 TEMAS ===')
    for i, g in enumerate(todos[:5]):
        print('#' + str(i+1) + '  ' + str(g['score']) + '  ' + g['group'])
    bos = [r for r in ar if r['breakout']]
    print('Rupturas: ' + str(len(bos)))
    print('')
    print('Web actualizada: https://Carolo-III.github.io/scanner-temas')
